In [ ]:
#Python 3.10.0

In [1]:
import numpy as np 
import pandas as pd
import os

In [2]:
data='/Users/Tysia/Desktop/Edukacja/Informatyka/Magisterka/ml-1m'
rating_data=os.path.join(data,'ratings.dat')
movie_data=os.path.join(data,'movies.dat')
user_data=os.path.join(data,'users.dat')

ratings=pd.read_csv(
  rating_data,
  sep='::',
  engine='python',
  names=['user','item','rating','timestamp'],
  encoding='latin-1'   
)

movies=pd.read_csv(
    movie_data, 
    sep='::',
    engine='python',
    names=['item', 'title', 'genres'],
    encoding='latin-1'
)
users = pd.read_csv(
    user_data, 
    sep='::',
    engine='python',
    names=['user', 'gender', 'age', 'occupation', 'zipcode'],
    encoding='latin-1'
)

In [ ]:
print(ratings.head())
print(movies.head())
print(users.head())

   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968
3        1      3408       4  978300275
4        1      2355       5  978824291
   movie_id                               title                        genres
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy
   user_id gender  age  occupation zipcode
0        1      F    1          10   48067
1        2      M   56          16   70072
2        3      M   25          15   55117
3        4      M   45           7   02460
4        5      M   25          20   55455


In [3]:
from surprise import Reader, Dataset, KNNBasic, accuracy, SVD
from surprise.model_selection import train_test_split as surprise_train_test_split

In [4]:
reader = Reader(rating_scale=(1, 5)) # skala movielens: 1-5

data = Dataset.load_from_df(ratings[['user', 'item', 'rating']], reader) #ważna kolejność kolumn

In [5]:
trainset, testset = surprise_train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
sim_options = {
    'k':'20',
    'name': 'cosine',       # miara podobieństwa - cosinus kąta, może też być pearson lub msd
    'user_based': False     # False = item-based, czyli podobieństwo przedmiotów
}

iknn= KNNBasic(sim_options=sim_options)
iknn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [7]:
predictions = iknn.test(testset)
rmse = accuracy.rmse(predictions) # jest jeszcze fcp, mse
mae = accuracy.mae(predictions)

RMSE: 0.9988
MAE:  0.7794


In [ ]:
sim_options = {
    'k':'20',
    'name': 'cosine',       # miara podobieństwa - cosinus kąta, może też być pearson lub msd
    'user_based': True     # podobieństwo użytkowników
}

uknn= KNNBasic(sim_options=sim_options)
uknn.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [7]:
predictions = uknn.test(testset)
rmse = accuracy.rmse(predictions) # jest jeszcze fcp, mse
mae = accuracy.mae(predictions)

RMSE: 0.9765
MAE:  0.7713


In [ ]:
# SVD uczy się ukrytych cech użytkowników i filmów
svd = SVD(n_factors=10, n_epochs=30, lr_all=0.01, reg_all=0.015, random_state=42) 
svd.fit(trainset)

In [11]:
predictions = svd.test(testset)
rmse = accuracy.rmse(predictions) # jest jeszcze mae, fcp, mse
mae = accuracy.mae(predictions)

rmse, mae

RMSE: 0.8641
MAE:  0.6740


(0.8641086993945439, 0.6740161288570656)